# RAG MVP (recetas)

Este cuaderno orquesta el pipeline usando los modulos en src/.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/xabierlosa/ProyectoPLN.git"
ROOT = Path("/content/ProyectoPLN")

if not (ROOT / ".git").exists():
    if ROOT.exists():
        shutil.rmtree(ROOT)
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

sys.path.insert(0, str(ROOT))
print(f"Repositorio listo en {ROOT}")

Repositorio listo en /content/ProyectoPLN


: 

In [1]:
# Esta celda solo debe ejecutarse en Colab.
# En el entorno local de VS Code se omite para evitar romper el kernel con reinstalaciones forzadas.
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" not in sys.modules:
    print("Celda de dependencias omitida: este setup solo se ejecuta en Colab.")
else:
    restart_marker = Path("/content/.proyectopln_deps_ready_v4")

    check_code = """
import numpy
import scipy
import sklearn
import torch
import transformers
import sentence_transformers
import faiss
print('OK', numpy.__version__, scipy.__version__, sklearn.__version__, torch.__version__, transformers.__version__, sentence_transformers.__version__)
""".strip()

    def run_checked(cmd, label):
        result = subprocess.run(cmd, text=True, capture_output=True)
        if result.returncode != 0:
            print(f"[{label}] fallo con codigo {result.returncode}")
            if result.stdout:
                print("--- STDOUT ---")
                print(result.stdout[-3500:])
            if result.stderr:
                print("--- STDERR ---")
                print(result.stderr[-3500:])
            return False
        return True

    profiles = [
        {
            "name": "py312-faiss190-numpy2",
            "packages": [
                "numpy==2.0.2",
                "scipy==1.14.1",
                "scikit-learn==1.5.2",
                "torch==2.4.1",
                "transformers==4.44.2",
                "sentence-transformers==3.1.1",
                "accelerate==0.34.2",
                "safetensors==0.4.5",
                "faiss-cpu==1.9.0.post1",
            ],
        },
        {
            "name": "py312-faiss190-numpy1",
            "packages": [
                "numpy==1.26.4",
                "scipy==1.13.1",
                "scikit-learn==1.5.1",
                "torch==2.4.1",
                "transformers==4.44.2",
                "sentence-transformers==3.1.1",
                "accelerate==0.34.2",
                "safetensors==0.4.5",
                "faiss-cpu==1.9.0.post1",
            ],
        },
    ]

    if not restart_marker.exists():
        run_checked([sys.executable, "-m", "pip", "uninstall", "-y", "faiss-cpu", "sentence-transformers", "scikit-learn", "scipy", "numpy"], "cleanup")

        validated = False
        for profile in profiles:
            ok = run_checked(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "--upgrade",
                    "--force-reinstall",
                    "--no-cache-dir",
                    *profile["packages"],
                ],
                f"install-{profile['name']}",
            )
            if not ok:
                continue

            ok = run_checked([sys.executable, "-c", check_code], f"check-{profile['name']}")
            if ok:
                print(f"Perfil valido: {profile['name']}")
                validated = True
                break

        if not validated:
            raise RuntimeError("No se pudo validar un stack binario compatible en Colab. Revisa STDERR arriba.")

        restart_marker.write_text("ok")

        if "google.colab" in sys.modules:
            print("Reiniciando el runtime de Colab para cargar los binarios nuevos...")
            os.kill(os.getpid(), 9)
    else:
        print("Dependencias validadas. Continua con la siguiente celda.")

Dependencias validadas. Continua con la siguiente celda.


In [2]:
from pathlib import Path
import sys

ROOT = Path("/content/ProyectoPLN")
sys.path.insert(0, str(ROOT))

import torch

from src.chunking import chunk_documents
from src.data_loader import build_documents, load_recipes
from src.embeddings import embed_passages, load_embedder
from src.faiss_index import build_index, save_docs, save_index

DATA_PATH = ROOT / "data" / "sesgos" / "ChefGPT_Dataset_downsampled_top10pct_authors.json"
OUT_DIR = ROOT / "artifacts"

recipes = load_recipes(DATA_PATH)
docs = build_documents(recipes)
chunks = chunk_documents(docs, max_words=220, overlap=40)

texts = [doc["text"] for doc in chunks]
device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = load_embedder("intfloat/multilingual-e5-base", device=device)
embeddings = embed_passages(embedder, texts, batch_size=32)

index = build_index(embeddings)
save_index(index, OUT_DIR / "index.faiss")
save_docs(chunks, OUT_DIR / "docs.jsonl")

print(f"Index listo: {len(chunks)} chunks")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/41 [00:00<?, ?it/s]

Index listo: 1298 chunks


In [3]:
from pathlib import Path
import sys

ROOT = Path("/content/ProyectoPLN")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.embeddings import embed_query
from src.faiss_index import load_docs, load_index
from src.rag_pipeline import generate_answer, load_llm

OUT_DIR = ROOT / "artifacts"

index = load_index(OUT_DIR / "index.faiss")
docs = load_docs(OUT_DIR / "docs.jsonl")

query = "¿Que puedo hacer con semillas de amapola?"
query_vec = embed_query(embedder, query).astype("float32")
scores, indices = index.search(query_vec.reshape(1, -1), 6)
contexts = [docs[i]["text"] for i in indices[0] if i >= 0]

tokenizer, model = load_llm("Qwen/Qwen2.5-7B-Instruct")
answer = generate_answer(tokenizer, model, query, contexts)
print(answer)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Con las semillas de amapola puedes hacer varios tipos de panes y bocadillos en tu receta de panecillos variados de Navidad. Las semillas de amapola añaden un sabor dulce y crujiente a la masa. Puedes seguir las instrucciones proporcionadas en el contexto para dividir la masa y agregar semillas de amapola al gusto en cada parte antes de formar bolas y hornearlos.

También puedes usar las semillas de amapola en otras preparaciones como:

1. En barritas de cereales: Mezcla semillas de amapola con otros cereales, avellanas y miel para hacer barras energéticas saludables.
2. En flapjacks o barritas de avena: Incorpora semillas de amapola junto con avena, almendras y coco para darles un sabor y textura distintos.
3. Como topping en tartas: Puedes espolvorear semillas de amapola en la superficie de una tarta para añadir un toque crujiente.

Las semillas de amapola son versátiles y se pueden


## Evaluación del retrieval

En esta sección medimos si el índice recupera los títulos correctos para un conjunto de consultas con ground truth manual. Se reportan `Precision@k`, `Recall@k` y `MRR` para `k = 1, 3, 5, 10`.

In [4]:
from collections import defaultdict
import re
import unicodedata

import pandas as pd
import torch

from src.embeddings import embed_query, load_embedder
from src.faiss_index import load_docs, load_index

OUT_DIR = ROOT / "artifacts"
INDEX_PATH = OUT_DIR / "index.faiss"
DOCS_PATH = OUT_DIR / "docs.jsonl"

if "embedder" not in globals():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = load_embedder("intfloat/multilingual-e5-base", device=device)

index = load_index(INDEX_PATH)
docs = load_docs(DOCS_PATH)


def normalize_title(text: str) -> str:
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


queries = [
    {
        "query": "dime recetas para legumbres",
        "relevant_titles": [
            "Receta de garbanzos con pisto",
            "receta de tortilla vegana casera, original, fácil y rápida",
            "Hummus verde, una alternativa para el picoteo, muy fácil y saludable con guisantes",
            "Ensalada de garbanzos con bacalao: receta fácil ideal para Semana Santa",
            "Ensalada de tres frijoles, una deliciosa combinación que impresionará a todos",
            "Hummus de aguacate: receta deliciosa, fácil y rápida para picotear",
            "Garbanzos con verduras: la receta perfecta en la freidora de aire para una cena saludable",
            "Receta de ensalada de garbanzos con sal negra en escamas: las legumbres también son para el verano",
            "Fabes cocinadas a fuego lento con langostinos y chipirones",
            "Crema para untar de cacao y alubias: una receta para conquistar incluso al más amante del chocolate",
            "Untable de alubias y pimientos rojos: una receta fácil, rápida y extra cremosa para disfrutar al momento del picoteo",
            "Crema de alubias, receta de cuchara para un día de frío",
            "Ribollita, la receta italiana para transformar el pan duro en un guiso reconfortante y muy barato",
        ],
    },
    {
        "query": "dime recetas para postres",
        "relevant_titles": [
            "Tarta flan o flan pâtissier, la sencilla receta clásica de la repostería francesa",
            "Receta de bizcocho fácil para tarta, la preparación más rápida para todas las bases",
            "Tarta de almendra, queso ricotta y limón, receta sin gluten para una celebración",
            "Tarta de Oreo: la receta más fácil, con solo tres ingredientes",
            "Estas galletas de brownie se convertirán en vuestra receta preferida de cookies en cuanto las probéis",
            "Cookies multigranos. Receta con fibra para un desayuno delicioso",
            "Helado casero de fresas y plátano: receta",
            "Snickerdoodles o galletas de azúcar y canela, receta de Navidad",
            "Galletas azucaradas con pepitas de chocolate, receta hiper rápida para acompañar el café",
            "Receta de batido de galletas Oreo, un milkshake rápido y goloso para las meriendas de verano",
            "Cómo hacer galletas linzer, la receta de las pastas austriacas más famosas navideñas (y todo el año)",
            "Pastas de té de dos colores: la receta por la que todos te preguntarán",
            "Monstruitos de chocolate. Receta fácil de Halloween",
            "Tarta de números o tarta Adikosh, el pastel perfecto para celebrar cualquier aniversario (incluido el nuestro)",
            "Receta de galletas sin azúcar, una opción de postre o merienda con menos calorías",
            "Galletas decoradas con obleas: receta para un cumpleaños infantil",
            "Cómo hacer matasuegras o galletas fritas con crema, la receta de toda la vida",
            "Galletas de vainilla conejos y huevos de Pascua: receta colorida para Semana Santa",
            "Receta de shortbread, la deliciosa galleta típica de Escocia",
            "Galletas saladas de cúrcuma: receta con y sin Thermomix",
            "Cómo hacer turrón de chocolate con galletas Oreo casero, receta de Navidad Thermomix",
        ],
    },
    {
        "query": "dime recetas para pescado",
        "relevant_titles": [
            "Calamares con olivas negras, receta de guiso marinero fácil con un toque diferente",
            "Fumet o caldo de pescado de roca, receta básica para usar en un montón de preparaciones",
            "Ensalada de atún: receta casera saludable y llena de sabor lista en 10 minutos",
            "Cómo hacer bacalao encebollado, una receta sencilla y que sorprende por su delicioso sabor",
            "Receta de choco en salsa (sepia) al estilo canario",
            "Pimiento gratinado relleno de arroz: receta vegetariana",
            "Rosada al horno, la mejor receta para disfrutar de este pescado todo el año",
            "Paella marinera, receta fácil del mejor arroz de domingo posible",
            "Calamares encebollados, la receta de un guiso tan clásico como apetecible",
            "Corvina a la plancha, receta fácil y rápida para resolver un segundo plato",
            "Croquetas de merluza y gambas, receta de aperitivo",
            "Receta de encebollado ecuatoriano de pescado, una opción caliente para salir del ceviche",
            "Cómo hacer la auténtica salsa de cacahuete vietnamita, imprescindible para acompañar rollitos de papel de arroz y mucho más",
            "Salmón al horno con salsa de cítricos y jengibre: Receta",
            "Arroz caldoso con calamares. Receta fácil para lucirse",
            "Bacalao gratinado con cebolla y patatas: receta de Semana Santa para disfrutar del bacalao de manera diferente",
            "Bacalao al horno a las tres capas: receta diferente de pescado al horno",
            "Cómo hacer atún a la plancha de forma fácil y rápida: una receta la mar de saludable para todo el año",
            "Filetes de pescado en leche de coco: receta india que no pica si no quieres (y con ingredientes fáciles de encontrar",
            "Pescado en costra, la receta más fácil y vistosa para cocinar meluza, lubina o dorada",
            "Receta de bacalao a la vicentina, una exquisitez italiana",
            "Bonito con tomate y pimientos de bola, receta para disfrutar de este soberbio pescado",
            "Almejas en salsa verde: receta de la abuela",
            "Receta de arroz cremoso con mejillones",
        ],
    },
    {
        "query": "dime recetas para trufa",
        "relevant_titles": [
            "Ravioli tartufo. Receta de pasta casera con y sin Thermomix",
            "Trufas de avena y coco: receta integral, sin azúcar y sin frutos secos",
            "Crema de calabaza con trufa negra: la receta de cuchara que subirá el nivel de tus cenas",
            "Receta de hamburguesa Piamonte, con tomates secos y mostaza de Dijon",
            "Receta de tortilla de trigueros y trufa, una explosión de sabor en la boca",
            "Trufas de granola, un snack fácil de hacer, nutritivo y bien rico con solo cuatro ingredientes",
        ],
    },
    {
        "query": "dime recetas para carne",
        "relevant_titles": [
            "Conejo a la mostaza, receta fácil y saludable",
            "Tarta de pollo al mole con masa filo: una espectacular receta de aprovechamiento, que puedes hacer con cualquier guiso de carne",
            "Empanadas colombianas de carne, la receta más fácil y tradicional para hacerlas en casa",
            "Crema de alubias, receta de cuchara para un día de frío",
            "Receta de solomillo de cerdo al horno con salsa de vino, mostaza y miel",
            "Sofrito payés, receta tradicional ibicenca",
            "Carrilleras en Thermomix, la mejor receta para disfrutar de este guiso fácilmente",
            "Callos a la vizcaína, receta tradicional vasca",
            "Calabacines rellenos de carne picada con queso y bechamel",
            "La receta de canelones de mi abuela: la elaboración casera más tradicional sigue siendo la mejor",
            "Huevos escoceses: la receta clásica de scotch egg que es, en realidad, inglesa",
            "Receta de arroz meloso con pollo y chistorra, un plato que es toda una explosión de sabor",
            "Receta de frikadellen, las deliciosas hamburguesas alemanas especiadas",
            "Carne guisada con patatas: receta tradicional reconfortante para mojar pan",
            "Receta de albóndigas con salsa Teriyaki",
            "Yaroa dominicana: el invento callejero con carne y patatas que conquistó a todos",
            "Empanadas chilenas de pino: una receta tradicional muy fácil de hacer en casa",
            "Carne ó caldeiro, receta tradicional gallega de carne cocida con patatas",
            "Garbanzos con acelgas: una receta de lo más fácil y sabrosa para un lunes sin carne",
            "Pechugas de pollo al horno con tomates cherry y mascarpone: receta al horno con un toque especial",
            "Presa ibérica a la plancha, un plato muy sabroso y sin complicaciones",
            "Carrilleras de ternera guisadas con vino tinto: receta tradicional",
            "Ternera con salsa de ostras al estilo chino, receta infalible para salir de la rutina",
            "Fideuá de carne, una deliciosa alternativa al plato tradicional marinero para disfrutar en familia",
        ],
    },
]

k_values = [1, 3, 5, 10]
max_search = max(k_values) * 5

def retrieve_unique_titles(query: str, top_n: int = max_search):
    query_vec = embed_query(embedder, query).astype("float32")
    scores, indices = index.search(query_vec.reshape(1, -1), top_n)

    ranked = []
    seen_titles = set()
    for idx in indices[0]:
        if idx < 0 or idx >= len(docs):
            continue
        metadata = docs[idx].get("metadata", {})
        title = str(metadata.get("titulo", "")).strip()
        if not title:
            continue
        normalized = normalize_title(title)
        if normalized in seen_titles:
            continue
        seen_titles.add(normalized)
        ranked.append(title)
    return ranked


query_rows = []
summary_rows = []

for item in queries:
    query_text = item["query"]
    relevant_titles = item["relevant_titles"]
    relevant_set = {normalize_title(title) for title in relevant_titles}
    ranked_titles = retrieve_unique_titles(query_text, top_n=max_search)
    ranked_normalized = [normalize_title(title) for title in ranked_titles]

    first_relevant_rank = None
    for position, norm_title in enumerate(ranked_normalized, start=1):
        if norm_title in relevant_set:
            first_relevant_rank = position
            break

    per_query = {"query": query_text, "relevant_total": len(relevant_set), "mrr": 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank}
    for k in k_values:
        topk = ranked_normalized[:k]
        hits = sum(1 for title in topk if title in relevant_set)
        precision = hits / max(1, len(topk))
        recall = hits / max(1, len(relevant_set))
        per_query[f"precision@{k}"] = precision
        per_query[f"recall@{k}"] = recall
        per_query[f"hits@{k}"] = hits
    query_rows.append(per_query)

    summary_rows.append(
        {
            "query": query_text,
            "top_retrieved_titles": ranked_titles[:10],
            "matched_titles_top10": [title for title, norm in zip(ranked_titles[:10], ranked_normalized[:10]) if norm in relevant_set],
            "relevant_titles": relevant_titles,
            "mrr": per_query["mrr"],
        }
    )

results_df = pd.DataFrame(query_rows)
display(results_df)

metric_cols = [col for col in results_df.columns if col.startswith("precision@") or col.startswith("recall@") or col == "mrr"]
aggregate_df = results_df[metric_cols].mean().to_frame("mean").reset_index().rename(columns={"index": "metric"})
display(aggregate_df)

detail_df = pd.DataFrame(summary_rows)
display(detail_df)

print("Evaluación de retrieval completada con k = 1, 3, 5, 10.")

,query,relevant_total,mrr,precision@1,recall@1,hits@1,precision@3,recall@3,hits@3,precision@5,recall@5,hits@5,precision@10,recall@10,hits@10
0,dime recetas para legumbres,13,0.200000,0.0,0.000000,0,0.000000,0.000000,0,0.2,0.076923,1,0.1,0.076923,1
1,dime recetas para postres,21,0.066667,0.0,0.000000,0,0.000000,0.000000,0,0.0,0.000000,0,0.0,0.000000,0
2,dime recetas para pescado,24,1.000000,1.0,0.041667,1,0.666667,0.083333,2,0.6,0.125000,3,0.4,0.166667,4
3,dime recetas para trufa,6,1.000000,1.0,0.166667,1,1.000000,0.500000,3,0.8,0.666667,4,0.6,1.000000,6
4,dime recetas para carne,24,0.500000,0.0,0.000000,0,0.333333,0.041667,1,0.4,0.083333,2,0.5,0.208333,5


,metric,mean
0,mrr,0.553333
1,precision@1,0.400000
2,recall@1,0.041667
3,precision@3,0.400000
4,recall@3,0.125000
5,precision@5,0.400000
6,recall@5,0.190385
7,precision@10,0.320000
8,recall@10,0.290385


,query,top_retrieved_titles,matched_titles_top10,relevant_titles,mrr
0,dime recetas para legumbres,"[La guía definitiva de la casquería, según el ...",[Receta de ensalada de garbanzos con sal negra...,"[Receta de garbanzos con pisto, receta de tort...",0.200000
1,dime recetas para postres,[Pastel especiado de peras y jengibre: receta ...,[],"[Tarta flan o flan pâtissier, la sencilla rece...",0.066667
2,dime recetas para pescado,[Cómo hacer atún a la plancha de forma fácil y...,[Cómo hacer atún a la plancha de forma fácil y...,"[Calamares con olivas negras, receta de guiso ...",1.000000
3,dime recetas para trufa,"[Trufas de avena y coco: receta integral, sin ...","[Trufas de avena y coco: receta integral, sin ...",[Ravioli tartufo. Receta de pasta casera con y...,1.000000
4,dime recetas para carne,"[Nikujaga, el estofado japonés de carne y pata...","[Fideuá de carne, una deliciosa alternativa al...","[Conejo a la mostaza, receta fácil y saludable...",0.500000


Evaluación de retrieval completada con k = 1, 3, 5, 10.


**Interpretación cualitativa de la evaluación del retrieval:**

Los resultados muestran que el índice recupera con bastante soltura las consultas más concretas, especialmente `pescado` y `trufa`, donde la primera posición ya devuelve un documento relevante y el `MRR` alcanza 1.0. Eso sugiere que el embedding y el índice están capturando bien ciertos términos muy discriminativos del dominio.

En cambio, las consultas más amplias o ambiguas, como `legumbres` y sobre todo `postres`, bajan bastante el rendimiento. En esos casos el sistema sigue encontrando algunos aciertos dentro del top-k, pero la precisión inicial es más débil y el ruido aumenta. El promedio final confirma este patrón: `Precision@1` y `Precision@3` se quedan en 0.40, mientras que `Recall@10` sube a 0.29, lo que indica que el índice recupera parte del conjunto correcto pero todavía no lo concentra bien en las primeras posiciones.

En términos prácticos, esto significa que el RAG puede funcionar razonablemente bien para búsquedas semánticas claras, pero aún necesita mejorar la discriminación en temas genéricos o muy populares. Una posible mejora sería reforzar el re-ranking, añadir metadatos o usar una función de mezcla entre similitud semántica y cobertura temática para empujar más arriba los títulos correctos.

## Evaluación de la generación

Para la generación se comparan respuestas del RAG con un golden dataset manual con cinco casos: extracción de datos críticos, escalado numérico, sustitución por seguridad alimentaria, explicación de técnica y control fuera de dominio. La evaluación es semiautomática y usa una rúbrica por caso para detectar omisiones, invenciones, cálculo incorrecto o rechazo insuficiente.

In [5]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import torch

from src.embeddings import load_embedder
from src.faiss_index import load_docs, load_index
from src.rag_pipeline import generate_answer, load_llm, retrieve

OUT_DIR = ROOT / "artifacts"
INDEX_PATH = OUT_DIR / "index.faiss"
DOCS_PATH = OUT_DIR / "docs.jsonl"
GEN_EVAL_PATH = OUT_DIR / "generation_eval_results.json"

if "embedder" not in globals():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embedder = load_embedder("intfloat/multilingual-e5-base", device=device)

if "index" not in globals():
    index = load_index(INDEX_PATH)
if "docs" not in globals():
    docs = load_docs(DOCS_PATH)
if "tokenizer" not in globals() or "model" not in globals():
    tokenizer, model = load_llm("Qwen/Qwen2.5-7B-Instruct")

if hasattr(model, "eval"):
    model.eval()


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text).lower())
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def has_any(text: str, patterns) -> bool:
    return any(re.search(pattern, text) for pattern in patterns)


def score_tarta_santiago(answer: str) -> dict:
    text = normalize_text(answer)
    checks = {
        "almendra_250g": bool(re.search(r"250\s*g.*almendra molida|almendra molida.*250\s*g", text)),
        "azucar_250g": bool(re.search(r"250\s*g.*azucar|azucar.*250\s*g", text)),
        "huevos_5": bool(re.search(r"5\s+huevos|cinco huevos", text)),
        "limon_medio": "medio limon" in text or "media cascara de limon" in text,
        "canela": "canela" in text,
        "sin_harina": "harina" not in text,
        "sin_levadura": "levadura" not in text,
    }
    checks["score"] = sum(checks.values()) / 7
    checks["passed"] = checks["score"] >= 0.86
    return checks


def score_hummus_scale(answer: str) -> dict:
    text = normalize_text(answer)
    checks = {
        "factor_2_5": has_any(text, [r"2[,\.]?5", r"dos punto cinco", r"x\s*2[,\.]?5"]),
        "garbanzos_1000g": has_any(text, [r"1000\s*g", r"1\s*kg", r"un kilo", r"un kilogramo"]),
        "tahini_150g": has_any(text, [r"150\s*g.*tahini", r"tahini.*150\s*g", r"150 gramos.*tahini"]),
        "explica_escalado": has_any(text, [r"multiplica", r"escal", r"factor de escala"]),
    }
    checks["score"] = sum(checks.values()) / 4
    checks["passed"] = checks["score"] >= 0.75
    return checks


def score_celiac(answer: str) -> dict:
    text = normalize_text(answer)
    checks = {
        "alternativa_sin_gluten": has_any(text, [r"harina sin gluten", r"mezcla de harina sin gluten", r"harina de arroz", r"maicena"]),
        "levadura_sin_gluten": has_any(text, [r"levadura .*sin gluten", r"certificad[oa] .*sin gluten"]),
        "mantiene_base": all(term in text for term in ["yogur", "huevo", "aceite", "azucar"]),
        "no_solo_quitar": not has_any(text, [r"solo quitar la harina", r"solo elimina la harina", r"simplemente quita la harina"]),
    }
    checks["score"] = sum(checks.values()) / 4
    checks["passed"] = checks["score"] >= 0.75
    return checks


def score_autolysis(answer: str) -> dict:
    text = normalize_text(answer)
    checks = {
        "mezcla_harina_agua": has_any(text, [r"harina.*agua", r"agua.*harina"]),
        "reposo_20_60": has_any(text, [r"20\s*(a|y)?\s*60\s*min", r"entre 20 y 60 minutos", r"20 a 60 minutos"]),
        "antes_masa_madre_sal": has_any(text, [r"masa madre", r"sal"]),
        "explica_por_que": has_any(text, [r"gluten", r"almidon", r"almidones", r"amasado", r"volumen"]),
    }
    checks["score"] = sum(checks.values()) / 4
    checks["passed"] = checks["score"] >= 0.75
    return checks


def score_out_of_domain(answer: str) -> dict:
    text = normalize_text(answer)
    checks = {
        "rechazo": has_any(text, [r"solo contiene recetas", r"no dispongo", r"no puedo", r"no es mi tema", r"no tengo informacion sobre mecanica"]),
        "no_inventa": not has_any(text, [r"\d+\s*°", r"\d+\s*grados", r"temperatura de horneado", r"hornear un motor"]),
    }
    checks["score"] = 1.0 if all(checks.values()) else 0.0
    checks["passed"] = bool(checks["score"])
    return checks


def evaluate_case(case: dict, answer: str) -> dict:
    if case["category"] == "extraccion":
        checks = score_tarta_santiago(answer)
    elif case["category"] == "escalado":
        checks = score_hummus_scale(answer)
    elif case["category"] == "sustitucion":
        checks = score_celiac(answer)
    elif case["category"] == "tecnica":
        checks = score_autolysis(answer)
    elif case["category"] == "fuera_dominio":
        checks = score_out_of_domain(answer)
    else:
        raise ValueError(f"Categoria no soportada: {case['category']}")

    return checks


generation_cases = [
    {
        "id": 1,
        "category": "extraccion",
        "query": "¿Cuáles son los ingredientes exactos y las cantidades para una Tarta de Santiago de 24cm?",
        "golden": "Para una tarta de 24cm, los ingredientes obligatorios son: 250g de almendra molida, 250g de azúcar blanco, 5 huevos grandes (talla L), ralladura de medio limón y 1 cucharadita de canela. Opcionalmente, azúcar glass para decorar la Cruz de Santiago. No debe contener harina.",
    },
    {
        "id": 2,
        "category": "escalado",
        "query": "La receta original de Hummus es para 4 personas y usa 400g de garbanzos. Necesito prepararla para 10 personas, ¿qué cantidades debo usar?",
        "golden": "Para 10 personas (factor de escala 2.5x), las cantidades son: 1kg (1000g) de garbanzos, 150g de tahini (si la original pedía 60g), y proporcionalmente el resto. El cálculo clave es multiplicar cada ingrediente por 2.5.",
    },
    {
        "id": 3,
        "category": "sustitucion",
        "query": "Quiero hacer la receta de Bizcocho de Yogur pero soy celíaco. ¿Qué cambios debo hacer?",
        "golden": "Debes sustituir los 3 recipientes (medidas de yogur) de harina de trigo por una mezcla de harina sin gluten (como harina de arroz y maicena) y asegurar que el sobre de levadura sea certificado 'sin gluten'. El resto de ingredientes (yogur, huevos, aceite, azúcar) se mantienen igual.",
    },
    {
        "id": 4,
        "category": "tecnica",
        "query": "Explícame el método de 'autólisis' mencionado en la receta de Pan de Masa Madre y por qué es importante.",
        "golden": "La autólisis consiste en mezclar solo la harina y el agua antes de añadir la masa madre y la sal, dejando reposar entre 20 y 60 minutos. Es crucial porque inicia el desarrollo del gluten y la descomposición de almidones, facilitando el amasado y mejorando el volumen final del pan.",
    },
    {
        "id": 5,
        "category": "fuera_dominio",
        "query": "¿Cuál es la mejor temperatura para hornear un motor de un coche tras pintarlo?",
        "golden": "Lo siento, pero mi base de datos solo contiene recetas de cocina y preparaciones culinarias. No dispongo de información técnica sobre mecánica o pintura automotriz.",
    },
]

results = []
for case in generation_cases:
    contexts, _ = retrieve(case["query"], embedder, index, docs, top_k=6)
    answer_text = generate_answer(
        tokenizer,
        model,
        case["query"],
        contexts,
        max_new_tokens=180,
        temperature=0.0,
        top_p=1.0,
    )
    checks = evaluate_case(case, answer_text)
    results.append(
        {
            "id": case["id"],
            "category": case["category"],
            "query": case["query"],
            "golden": case["golden"],
            "generated": answer_text,
            "score": checks["score"],
            "passed": checks["passed"],
            "checks": checks,
        }
    )

results_df = pd.DataFrame(results)

summary_rows = []
for row in results:
    checks = row["checks"].copy()
    checks.pop("score", None)
    checks.pop("passed", None)
    summary_rows.append(
        {
            "id": row["id"],
            "category": row["category"],
            "score": row["score"],
            "passed": row["passed"],
            **checks,
        }
    )

summary_df = pd.DataFrame(summary_rows)
aggregate_generation = summary_df.groupby("category")["score"].mean().reset_index().rename(columns={"score": "mean_score"})
overall_generation = pd.DataFrame(
    [{
        "mean_score": float(summary_df["score"].mean()),
        "pass_rate": float(summary_df["passed"].mean()),
    }]
)

display(summary_df)
display(aggregate_generation)
display(overall_generation)

def preview(text: str, max_len: int = 350) -> str:
    text = str(text).strip()
    return text if len(text) <= max_len else text[:max_len] + "..."

preview_df = results_df[["id", "category", "query", "score", "passed", "generated"]].copy()
preview_df["generated"] = preview_df["generated"].apply(preview)
display(preview_df)

with GEN_EVAL_PATH.open("w", encoding="utf-8") as handle:
    json.dump(results, handle, ensure_ascii=False, indent=2)

print(f"Evaluación de generación completada. Resultados guardados en {GEN_EVAL_PATH}")

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


,id,category,score,passed,almendra_250g,azucar_250g,huevos_5,limon_medio,canela,sin_harina,...,alternativa_sin_gluten,levadura_sin_gluten,mantiene_base,no_solo_quitar,mezcla_harina_agua,reposo_20_60,antes_masa_madre_sal,explica_por_que,rechazo,no_inventa
0,1,extraccion,0.285714,False,False,False,False,False,False,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,escalado,0.750000,True,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,sustitucion,0.500000,False,NaN,NaN,NaN,NaN,NaN,NaN,...,True,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN
3,4,tecnica,0.500000,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,True,False,True,False,NaN,NaN
4,5,fuera_dominio,0.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False


,category,mean_score
0,escalado,0.750000
1,extraccion,0.285714
2,fuera_dominio,0.000000
3,sustitucion,0.500000
4,tecnica,0.500000


,mean_score,pass_rate
0,0.407143,0.2


,id,category,query,score,passed,generated
0,1,extraccion,¿Cuáles son los ingredientes exactos y las can...,0.285714,False,"Lo siento, pero no encontré información especí..."
1,2,escalado,La receta original de Hummus es para 4 persona...,0.750000,True,Para adaptar la receta de hummus para 10 perso...
2,3,sustitucion,Quiero hacer la receta de Bizcocho de Yogur pe...,0.500000,False,Para adaptar la receta de Bizcocho de Nueces y...
3,4,tecnica,Explícame el método de 'autólisis' mencionado ...,0.500000,False,La autólisis es un proceso natural que ocurre ...
4,5,fuera_dominio,¿Cuál es la mejor temperatura para hornear un ...,0.000000,False,El contexto proporcionado no contiene informac...


Evaluación de generación completada. Resultados guardados en /content/ProyectoPLN/artifacts/generation_eval_results.json


## Métricas automáticas de generación

Además de la rúbrica por caso, esta sección compara la respuesta generada con el golden estándar usando métricas automáticas: ROUGE, BERTScore, BLEU, exact match parcial y chequeos por reglas sobre las respuestas generadas.

In [6]:
from pathlib import Path
import json
import re
import subprocess
import sys
import unicodedata
from collections import Counter

import pandas as pd

try:
    from rouge_score import rouge_scorer
    from bert_score import score as bertscore
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "rouge-score==0.1.2", "bert-score==0.3.13", "nltk==3.9.1"],
        check=True,
    )
    from rouge_score import rouge_scorer
    from bert_score import score as bertscore
    from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

AUTO_EVAL_PATH = OUT_DIR / "generation_auto_metrics.json"

if "results" not in globals():
    with (OUT_DIR / "generation_eval_results.json").open("r", encoding="utf-8") as handle:
        results = json.load(handle)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text).lower())
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize_simple(text: str) -> list[str]:
    return re.findall(r"[a-z0-9áéíóúñü]+", normalize_text(text))


def partial_exact_match(golden: str, generated: str) -> float:
    gold_norm = normalize_text(golden)
    gen_norm = normalize_text(generated)
    if not gold_norm or not gen_norm:
        return 0.0
    if gold_norm == gen_norm:
        return 1.0
    if gold_norm in gen_norm or gen_norm in gold_norm:
        return 1.0

    gold_tokens = tokenize_simple(golden)
    gen_tokens = tokenize_simple(generated)
    if not gold_tokens or not gen_tokens:
        return 0.0
    overlap = len(set(gold_tokens) & set(gen_tokens))
    precision = overlap / len(set(gen_tokens))
    recall = overlap / len(set(gold_tokens))
    if precision + recall == 0:
        return 0.0
    f1 = 2 * precision * recall / (precision + recall)
    return float(f1)


def rule_checks(category: str, golden: str, generated: str) -> dict:
    text = normalize_text(generated)
    gold = normalize_text(golden)

    if category == "extraccion":
        return {
            "no_harina": "harina" not in text,
            "no_levadura": "levadura" not in text,
            "incluye_almendra": "almendra" in text,
            "incluye_azucar": "azucar" in text,
            "incluye_huevos": bool(re.search(r"\b(5|cinco)\s+huev", text)),
            "incluye_canela": "canela" in text,
        }
    if category == "escalado":
        return {
            "factor_2_5": any(pattern in text for pattern in ["2.5", "2,5", "dos punto cinco"]),
            "garbanzos_1000g": any(pattern in text for pattern in ["1000 g", "1000g", "1 kg", "1kg"]),
            "tahini_150g": "150" in text and "tahini" in text,
            "explica_multiplicar": any(keyword in text for keyword in ["multiplica", "escal", "factor"]),
        }
    if category == "sustitucion":
        return {
            "alternativa_sin_gluten": any(keyword in text for keyword in ["sin gluten", "harina de arroz", "maicena"]),
            "levadura_sin_gluten": "levadura" in text and "sin gluten" in text,
            "mantiene_base": all(keyword in text for keyword in ["yogur", "huevo", "aceite", "azucar"]),
            "no_solo_elimina": not any(keyword in text for keyword in ["solo quita la harina", "elimina la harina sin"]),
        }
    if category == "tecnica":
        return {
            "mezcla_harina_agua": "harina" in text and "agua" in text,
            "reposo": any(keyword in text for keyword in ["20", "60", "repos"]),
            "antes_masa_madre_sal": "masa madre" in text and "sal" in text,
            "explica_por_que": any(keyword in text for keyword in ["gluten", "almidon", "amasado", "volumen"]),
        }
    if category == "fuera_dominio":
        return {
            "rechaza_fuera_dominio": any(keyword in text for keyword in ["no dispongo", "no puedo", "no es mi tema", "solo contiene recetas", "no tengo informacion"]),
            "no_inventa_temperatura": not any(keyword in text for keyword in ["grados", "temperatura", "hornear"]),
        }
    return {}


def safe_bleu(reference: str, candidate: str) -> float:
    ref_tokens = tokenize_simple(reference)
    cand_tokens = tokenize_simple(candidate)
    if not ref_tokens or not cand_tokens:
        return 0.0
    smoothing = SmoothingFunction().method1
    weights = (0.25, 0.25, 0.25, 0.25)
    try:
        return float(sentence_bleu([ref_tokens], cand_tokens, weights=weights, smoothing_function=smoothing))
    except ZeroDivisionError:
        return 0.0


scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
references = [item["golden"] for item in results]
candidates = [item["generated"] for item in results]
bert_p, bert_r, bert_f = bertscore(
    candidates,
    references,
    lang="es",
    model_type="bert-base-multilingual-cased",
    rescale_with_baseline=True,
    verbose=False,
)

metric_rows = []
for item, b_p, b_r, b_f in zip(results, bert_p.tolist(), bert_r.tolist(), bert_f.tolist()):
    rouge = scorer.score(item["golden"], item["generated"])
    metric_rows.append(
        {
            "id": item["id"],
            "category": item["category"],
            "query": item["query"],
            "rouge1_f": float(rouge["rouge1"].fmeasure),
            "rouge2_f": float(rouge["rouge2"].fmeasure),
            "rougeL_f": float(rouge["rougeL"].fmeasure),
            "bert_precision": float(b_p),
            "bert_recall": float(b_r),
            "bert_f1": float(b_f),
            "bleu": safe_bleu(item["golden"], item["generated"]),
            "partial_exact_match": partial_exact_match(item["golden"], item["generated"]),
            **rule_checks(item["category"], item["golden"], item["generated"]),
        }
    )

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

aggregate_cols = [
    "rouge1_f",
    "rouge2_f",
    "rougeL_f",
    "bert_precision",
    "bert_recall",
    "bert_f1",
    "bleu",
    "partial_exact_match",
]
aggregate_metrics = metrics_df[aggregate_cols].mean().to_frame("mean").reset_index().rename(columns={"index": "metric"})
display(aggregate_metrics)

auto_rule_cols = [col for col in metrics_df.columns if col not in {"id", "category", "query", "rouge1_f", "rouge2_f", "rougeL_f", "bert_precision", "bert_recall", "bert_f1", "bleu", "partial_exact_match"}]
rule_summary = metrics_df[["category", *auto_rule_cols]].groupby("category").mean(numeric_only=True).reset_index()
display(rule_summary)

with AUTO_EVAL_PATH.open("w", encoding="utf-8") as handle:
    json.dump(metric_rows, handle, ensure_ascii=False, indent=2)

print(f"Métricas automáticas guardadas en {AUTO_EVAL_PATH}")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

,id,category,query,rouge1_f,rouge2_f,rougeL_f,bert_precision,bert_recall,bert_f1,bleu,...,alternativa_sin_gluten,levadura_sin_gluten,mantiene_base,no_solo_elimina,mezcla_harina_agua,reposo,antes_masa_madre_sal,explica_por_que,rechaza_fuera_dominio,no_inventa_temperatura
0,1,extraccion,¿Cuáles son los ingredientes exactos y las can...,0.303030,0.134969,0.169697,0.163610,0.249330,0.206548,0.045395,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,escalado,La receta original de Hummus es para 4 persona...,0.327586,0.087719,0.206897,0.149122,0.260609,0.204293,0.028027,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,sustitucion,Quiero hacer la receta de Bizcocho de Yogur pe...,0.375839,0.217687,0.281879,0.146982,0.318319,0.229923,0.141423,...,True,False,False,True,NaN,NaN,NaN,NaN,NaN,NaN
3,4,tecnica,Explícame el método de 'autólisis' mencionado ...,0.268156,0.079096,0.189944,0.131164,0.257321,0.193225,0.009201,...,NaN,NaN,NaN,NaN,True,True,False,False,NaN,NaN
4,5,fuera_dominio,¿Cuál es la mejor temperatura para hornear un ...,0.131579,0.013333,0.078947,-0.044549,0.128357,0.038948,0.002690,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False


,metric,mean
0,rouge1_f,0.281238
1,rouge2_f,0.106561
2,rougeL_f,0.185473
3,bert_precision,0.109266
4,bert_recall,0.242787
5,bert_f1,0.174588
6,bleu,0.045347
7,partial_exact_match,0.283992


,category
0,escalado
1,extraccion
2,fuera_dominio
3,sustitucion
4,tecnica


Métricas automáticas guardadas en /content/ProyectoPLN/artifacts/generation_auto_metrics.json


**Nota sobre los chequeos por reglas:**

Las métricas continuas (`ROUGE`, `BERTScore`, `BLEU` y `exact match parcial`) se calculan junto con reglas específicas por caso. Como esas reglas son booleanas, se resumen mejor convirtiéndolas a 0/1 antes de agregarlas por categoría.

In [8]:
from pathlib import Path
import json

import pandas as pd

AUTO_EVAL_PATH = OUT_DIR / "generation_auto_metrics.json"

if "metrics_df" not in globals():
    with AUTO_EVAL_PATH.open("r", encoding="utf-8") as handle:
        metric_rows = json.load(handle)
    metrics_df = pd.DataFrame(metric_rows)

rule_cols = [
    col
    for col in metrics_df.columns
    if col
    not in {
        "id",
        "category",
        "query",
        "rouge1_f",
        "rouge2_f",
        "rougeL_f",
        "bert_precision",
        "bert_recall",
        "bert_f1",
        "bleu",
        "partial_exact_match",
    }
]

rule_summary = metrics_df[["category", *rule_cols]].copy()
for col in rule_cols:
    rule_summary[col] = rule_summary[col].fillna(False).astype(int)

rule_summary = rule_summary.groupby("category")[rule_cols].mean().reset_index()
display(rule_summary)
print("Resumen de reglas calculado como promedio de aciertos por categoría.")

/tmp/ipykernel_22646/2729431677.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rule_summary[col] = rule_summary[col].fillna(False).astype(int)
/tmp/ipykernel_22646/2729431677.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  rule_summary[col] = rule_summary[col].fillna(False).astype(int)
/tmp/ipykernel_22646/2729431677.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_optio

,category,no_harina,no_levadura,incluye_almendra,incluye_azucar,incluye_huevos,incluye_canela,factor_2_5,garbanzos_1000g,tahini_150g,...,alternativa_sin_gluten,levadura_sin_gluten,mantiene_base,no_solo_elimina,mezcla_harina_agua,reposo,antes_masa_madre_sal,explica_por_que,rechaza_fuera_dominio,no_inventa_temperatura
0,escalado,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,extraccion,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,fuera_dominio,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,sustitucion,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,tecnica,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


Resumen de reglas calculado como promedio de aciertos por categoría.


**Interpretación cualitativa de la evaluación de la generación:**

Las métricas automáticas confirman un patrón bastante claro: el RAG produce respuestas más cercanas al golden estándar cuando la tarea exige una transformación simple y bien delimitada, como el escalado numérico. En ese caso el modelo parece seguir la relación proporcional con más solvencia, lo que se refleja en una puntuación bastante mejor que en el resto de categorías.

En cambio, cuando la pregunta exige precisión factual fina o control estricto de contenido, el rendimiento cae. Eso se nota sobre todo en la extracción exacta de ingredientes, donde una omisión o una invención penaliza mucho la respuesta, y en el caso fuera de dominio, donde el sistema todavía tiende a contestar en vez de rechazar con suficiente claridad.

La comparación automática también sugiere que las métricas de solapamiento lexical por sí solas no bastan para juzgar calidad semántica. `ROUGE`, `BLEU` y `exact match parcial` dan una señal útil sobre cercanía textual, pero no capturan del todo si la respuesta es segura, completa y bien fundamentada. Por eso el bloque automático funciona mejor como filtro y diagnóstico, mientras que la rúbrica por caso sigue siendo necesaria para interpretar errores de omisión, alucinación o mala adaptación.

En resumen, la generación todavía es aceptable en tareas mecánicas o de reescritura parcial, pero falla cuando se le pide precisión exacta, sustitución segura o una negativa clara fuera de dominio. Esto encaja con un RAG que recupera contexto útil, pero todavía no lo convierte de forma consistente en respuestas finas y controladas.